# MayDay — Train Urticaria trên Colab T4
Notebook chưa được chạy trên Colab/GPU. Đã kiểm tra cú pháp và chuyển nhãn trên máy local.

1. Upload file `Urticaria.v2i.yolov8.zip` lên Google Drive tại `MyDrive/MayDayAI/`.
2. Trong Colab chọn **Runtime → Change runtime type → T4 GPU**.
3. Chạy lần lượt từng ô. Notebook giữ ZIP gốc, tạo bản detection riêng trong `/content`.

Dataset có 100 ảnh (69/21/10), 3 lớp Body/Multiple/Single và nhãn trộn box/polygon. Notebook chuyển polygon thành box để train YOLOv8n detection; không giả lập mask cho box và không chạy model segmentation. `Multiple` là một vùng gộp, không cho biết chính xác bao nhiêu nốt. Không dùng tổng số box làm số nốt thật hoặc tự suy ra AUAS. Chưa loại trừ trùng bệnh nhân giữa các split vì thiếu mã bệnh nhân.


In [ ]:
import sys, subprocess, json
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Chọn runtime T4 GPU rồi chạy lại.'
print('Python:', sys.version)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
# Giữ torch/CUDA có sẵn của Colab. Không tạo venv hay cài lại torch.
import importlib.metadata as metadata
def run_logged(command, *, log_path=None, **kwargs):
    """Stream stdout and stderr together; retain the complete error output."""
    if any(key in kwargs for key in ("stdout","stderr","capture_output","text")):
        raise ValueError("run_logged owns output capture")
    kwargs.pop("check",None)
    print("Running:"," ".join(map(str,command)),flush=True)
    process=subprocess.Popen(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1,**kwargs)
    chunks=[]
    with process.stdout:
        for line in process.stdout:
            print(line,end="",flush=True);chunks.append(line)
    code=process.wait();output="".join(chunks)
    if log_path is not None:
        with Path(log_path).open("a",encoding="utf-8") as stream:
            stream.write("COMMAND: "+repr(list(map(str,command)))+"\n"+output+"\nEXIT: "+str(code)+"\n")
    if code:raise subprocess.CalledProcessError(code,command,output=output)
    return subprocess.CompletedProcess(command,code,stdout=output)
constraints = Path('/content/mayday-runtime-constraints.txt')
versions = {n: metadata.version(n) for n in ('torch','torchvision','numpy')}
constraints.write_text(''.join(f'{n}=={v}\n' for n,v in versions.items()))
run_logged([sys.executable, '-m', 'pip', '--version'], log_path='/content/mayday-setup.log')
run_logged([sys.executable, '-m', 'pip', 'install', '--constraint', str(constraints), 'ultralytics==8.3.221'], log_path='/content/mayday-setup.log')
assert all(metadata.version(n)==v for n,v in versions.items()), 'Runtime libraries changed; stop and inspect setup log.'
run_logged([sys.executable, '-c', "import torch, torchvision, ultralytics; assert torch.cuda.is_available(); print(torch.__version__, torch.version.cuda, ultralytics.__version__); print((torch.ones(2, device='cuda')+1).cpu())"], log_path='/content/mayday-setup.log')
from ultralytics import YOLO


In [ ]:
from google.colab import drive
from datetime import datetime
import hashlib

drive.mount('/content/drive')
ZIP = Path('/content/drive/MyDrive/MayDayAI/Urticaria.v2i.yolov8.zip')
assert ZIP.is_file(), f'Chưa có file: {ZIP}'
DATASET_SHA = hashlib.sha256(ZIP.read_bytes()).hexdigest()
DATASET = Path('/content') / ('mayday_urticaria_det_' + DATASET_SHA[:12])
PROJECT = Path('/content/drive/MyDrive/MayDayAI/runs')
RUN_NAME = 'yolov8n_' + datetime.now().strftime('%Y%m%d_%H%M%S')
print('Dataset SHA256:', DATASET_SHA)


In [ ]:
PREPARATION_SOURCE = '"""Convert the supplied mixed box/polygon export into a separate detection dataset.\n\nOnly images and labels are read. No instructions or executable content from the ZIP\nare executed. The input archive is never modified.\n"""\nimport argparse\nimport collections\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport zipfile\n\nNAMES = ["Body", "Multiple", "Single"]\n\n\ndef detection_row(line):\n    values = [float(v) for v in line.split()]\n    if not values or not all(math.isfinite(v) for v in values):\n        raise ValueError("Empty or non-finite label")\n    cls = int(values[0])\n    if values[0] != cls or cls not in range(len(NAMES)):\n        raise ValueError("Unknown class")\n    coords = values[1:]\n    if any(v < 0 or v > 1 for v in coords):\n        raise ValueError("Coordinates must be normalized to [0,1]")\n    if len(values) == 5:\n        x, y, w, h = coords\n        kind = "box"\n    elif len(coords) >= 6 and len(coords) % 2 == 0:\n        xs, ys = coords[::2], coords[1::2]\n        x, y = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2\n        w, h = max(xs) - min(xs), max(ys) - min(ys)\n        kind = "polygon"\n    else:\n        raise ValueError("Expected xywh box or polygon with at least three points")\n    if w <= 0 or h <= 0:\n        raise ValueError("Degenerate annotation")\n    return f"{cls} {x:.10f} {y:.10f} {w:.10f} {h:.10f}", kind, cls\n\n\ndef prepare(archive, output):\n    archive, output = Path(archive), Path(output).resolve()\n    if output.exists():\n        raise FileExistsError(f"Use a new empty output path; refusing overwrite: {output}")\n    report = {"source_sha256": hashlib.sha256(archive.read_bytes()).hexdigest(),\n              "names": NAMES, "images": {}, "annotations": {}, "input_formats": {},\n              "exact_duplicate_images": [], "warning": "Multiple marks regions, not a known number of individual wheals. Patient identities are unavailable; patient-level split leakage has not been ruled out."}\n    files = {}\n    annotation_types, class_counts, hashes = collections.Counter(), collections.Counter(), {}\n    with zipfile.ZipFile(archive) as z:\n        for info in z.infolist():\n            p = PurePosixPath(info.filename)\n            if len(p.parts) != 3 or p.parts[0] not in ("train", "valid", "test"):\n                continue\n            split, folder, filename = p.parts\n            if folder not in ("images", "labels") or filename in (".", ".."):\n                continue\n            if folder == "images" and p.suffix.lower() not in (".jpg", ".jpeg", ".png"):\n                continue\n            if folder == "labels" and p.suffix.lower() != ".txt":\n                continue\n            if info.filename in files:\n                raise ValueError(f"Duplicate image/label member: {info.filename}")\n            files[info.filename] = z.read(info)\n        if not files:\n            raise ValueError("No train/valid/test images and labels found")\n        converted = {}\n        for split in ("train", "valid", "test"):\n            images = [n for n in files if n.startswith(f"{split}/images/")]\n            if not images:\n                raise ValueError(f"Missing images for {split}")\n            report["images"][split] = len(images)\n            for name in images:\n                content = files[name]\n                sha = hashlib.sha256(content).hexdigest()\n                if sha in hashes:\n                    report["exact_duplicate_images"].append([hashes[sha], name])\n                hashes[sha] = name\n                label = str(PurePosixPath(name.replace("/images/", "/labels/")).with_suffix(".txt"))\n                if label not in files:\n                    raise ValueError(f"Missing label: {label}; verify negatives explicitly")\n                rows = []\n                for line_number, line in enumerate(files[label].decode("utf-8-sig").splitlines(), 1):\n                    if not line.strip():\n                        continue\n                    try:\n                        row, kind, cls = detection_row(line)\n                    except ValueError as exc:\n                        raise ValueError(f"{label}:{line_number}: {exc}") from exc\n                    rows.append(row)\n                    annotation_types[kind] += 1\n                    class_counts[NAMES[cls]] += 1\n                converted[label] = ("\\n".join(rows) + "\\n").encode()\n        cross_split = [pair for pair in report["exact_duplicate_images"] if pair[0].split(\'/\')[0] != pair[1].split(\'/\')[0]]\n        if cross_split:\n            raise ValueError(f"Exact duplicate images across splits: {cross_split}")\n        output.mkdir(parents=True)\n        for name, content in files.items():\n            if "/images/" in name:\n                target = output / name\n                target.parent.mkdir(parents=True, exist_ok=True)\n                target.write_bytes(content)\n        for name, content in converted.items():\n            target = output / name\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(content)\n    report["input_formats"] = dict(annotation_types)\n    report["annotations"] = dict(class_counts)\n    config = f"path: {output.as_posix()}\\ntrain: train/images\\nval: valid/images\\ntest: test/images\\nnames:\\n" + "".join(f"  {i}: {name}\\n" for i, name in enumerate(NAMES))\n    (output / "data.yaml").write_text(config, encoding="utf-8")\n    (output / "audit.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    return report\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("archive")\n    parser.add_argument("output")\n    args = parser.parse_args()\n    print(json.dumps(prepare(args.archive, args.output), indent=2))\n'
namespace = {'__name__': 'mayday_prepare'}
exec(compile(PREPARATION_SOURCE, 'prepare_urticaria.py', 'exec'), namespace)
if DATASET.exists():
    raise RuntimeError('Thư mục dataset đã tồn tại. Giữ nguyên kết quả cũ; đổi DATASET sang đường dẫn mới hoặc dùng data.yaml cũ sau khi kiểm tra audit.json. Notebook không tự xóa dữ liệu.')
audit = namespace['prepare'](ZIP, DATASET)
print(json.dumps(audit, indent=2))


## Kiểm tra nhãn trước khi train
Xem vị trí box so với tổn thương. Polygon chuyển sang khung bao sẽ mất hình dạng biên; nó không sửa được nhãn thiếu hoặc nhầm. Body là vùng cơ thể, Multiple là vùng gộp, Single là nhãn đơn theo người gán nhãn; cần chuyên gia xác nhận ý nghĩa lâm sàng. Nếu nhãn sai, sửa bản dataset rồi train lại.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import random
images = sorted((DATASET/'train/images').glob('*'))
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
colors = ['cyan','orange','lime']
for ax, image_path in zip(axes.flat, random.Random(42).sample(images, min(6,len(images)))):
    image = Image.open(image_path).convert('RGB')
    w,h=image.size
    ax.imshow(image)
    label = DATASET/'train/labels'/(image_path.stem+'.txt')
    for line in label.read_text().splitlines():
        if not line.strip(): continue
        cls,x,y,bw,bh=map(float,line.split()); cls=int(cls)
        ax.add_patch(Rectangle(((x-bw/2)*w,(y-bh/2)*h),bw*w,bh*h,fill=False,edgecolor=colors[cls],linewidth=1))
    ax.set_title('Body=cyan, Multiple=orange, Single=green'); ax.axis('off')
plt.tight_layout()
plt.show()


## Train baseline
Dùng YOLOv8n pretrained, ảnh 640, batch 8, AMP FP16, tối đa 100 epoch; dừng sớm sau 20 epoch không cải thiện. Nếu hết VRAM, giảm batch xuống 4. Ảnh nguồn đã resize 480×480 nên tăng lên 640 không khôi phục chi tiết đã mất. Dataset được đọc từ ổ local, checkpoint được lưu vào Drive. Khi resume chỉ dùng `last.pt` do bạn đã train và tin cậy.


In [ ]:
import shutil, time
model=YOLO('yolov8n.pt')
started=time.perf_counter()
results=model.train(
    data=str(DATASET/'data.yaml'), epochs=100, patience=20,
    imgsz=640, batch=8, device=0, workers=2, amp=True,
    seed=42, deterministic=True, cache=False,
    project=str(PROJECT), name=RUN_NAME, exist_ok=False,
    save=True, save_period=5, plots=True, max_det=1000,
    hsv_h=0.005, hsv_s=0.1, hsv_v=0.2, mosaic=0.2,
    flipud=0.0, fliplr=0.5,
)
RUN_DIR=Path(model.trainer.save_dir)
print('Elapsed minutes:', round((time.perf_counter()-started)/60,2))
for src in [DATASET/'audit.json', DATASET/'data.yaml',Path('/content/mayday-setup.log')]:
    shutil.copy2(src,RUN_DIR/src.name)
(RUN_DIR/'prepare_urticaria.py').write_text(PREPARATION_SOURCE)
run_logged([sys.executable,'-m','pip','freeze'],log_path=RUN_DIR/'environment.txt')
print('Best:',RUN_DIR/'weights/best.pt')
print('Resume: YOLO(str(RUN_DIR/"weights/last.pt")).train(resume=True)')


In [ ]:
# Không chọn ngưỡng hoặc model dựa trên tập test; dùng validation cho lựa chọn đó.
best=YOLO(str(RUN_DIR/'weights/best.pt'))
metrics=best.val(data=str(DATASET/'data.yaml'),split='test',imgsz=640,batch=8,device=0,plots=True,max_det=1000,project=str(RUN_DIR),name='test')
print('mAP50:',metrics.box.map50,'mAP50-95:',metrics.box.map)
predictions=best.predict(source=str(DATASET/'test/images'),imgsz=640,conf=0.25,device=0,max_det=1000,save=True,save_txt=True,save_conf=True,project=str(RUN_DIR),name='test_predictions')
for p in predictions[:5]:
    counts={best.names[i]:int((p.boxes.cls==i).sum().item()) for i in range(3)}
    print(Path(p.path).name,counts)
print('Single boxes / Multiple regions / Body regions — không phải tổng số nốt lâm sàng.')


## Thử ảnh mới
Chụp ảnh da rõ, đủ sáng, không cần khoanh vùng. Upload ảnh thử lên Drive và sửa đường dẫn bên dưới. Không dùng ảnh tập test để chỉnh model liên tục. Chưa có kiểm tra chất lượng ảnh tự động hoặc công thức AUAS trong notebook này.


In [ ]:
NEW_IMAGE = Path('/content/drive/MyDrive/MayDayAI/my_skin_photo.jpg')
if NEW_IMAGE.is_file():
    outputs=best.predict(source=str(NEW_IMAGE),imgsz=640,conf=0.25,device=0,max_det=1000,save=True,project=str(RUN_DIR),name='new_images')
    from IPython.display import display
    display(Image.fromarray(outputs[0].plot()[...,::-1]))
else:
    print('Đặt ảnh thử tại',NEW_IMAGE,'rồi chạy lại ô này.')


## Dùng model trên máy cá nhân
Tải `weights/best.pt` và `args.yaml` từ Drive. Trên máy Intel Iris Xe, có thể chạy PyTorch bằng `device="cpu"`, `imgsz=640`, batch 1. Không cần GPU NVIDIA để inference CPU, nhưng chưa đo độ trễ trên máy này. Colab dùng cho train và thử nghiệm tương tác; không coi Colab free là máy chủ API luôn hoạt động.

Nguồn: https://docs.ultralytics.com/models/yolov8/ · https://docs.ultralytics.com/tasks/detect/ · https://research.google.com/colaboratory/faq.html
